# MedHELM Workshop - Exploring Scenarios

In this notebook, you'll learn about:
1. Available MedHELM scenarios
2. The clinician-validated taxonomy
3. How to run different scenarios
4. Comparing results across scenarios

## 1. MedHELM Taxonomy Overview

MedHELM evaluates models across a comprehensive taxonomy developed with clinicians:

### 5 Main Categories:

1. **Clinical Decision Support**
   - Supporting Diagnostic Decisions
   - Planning Treatments
   - Predicting Patient Risks and Outcomes
   - Providing Clinical Knowledge Support

2. **Clinical Note Generation**
   - Documenting Patient Visits
   - Recording Procedures
   - Documenting Diagnostic Reports
   - Documenting Care Plans

3. **Patient Communication and Education**
   - Providing Patient Education Resources
   - Delivering Personalized Care Instructions
   - Patient-Provider Messaging
   - Enhancing Patient Understanding
   - Facilitating Patient Engagement

4. **Medical Research Assistance**
   - Conducting Literature Research
   - Analyzing Clinical Research Data
   - Recording Research Processes
   - Ensuring Clinical Research Quality
   - Managing Research Enrollment

5. **Administration and Workflow**
   - Scheduling Resources and Staff
   - Overseeing Financial Activities
   - Organizing Workflow Processes
   - Care Coordination and Planning

## 2. Popular MedHELM Scenarios

Here are some commonly used scenarios in MedHELM:

### Public Scenarios (No authentication required):

| Scenario | Category | Task | Instances |
|----------|----------|------|----------|
| **pubmed_qa** | Research Assistance | Literature QA | ~1,000 |
| **med_qa** | Clinical Knowledge | Medical Exam Questions | ~1,273 |
| **mimic_discharge** | Note Generation | Discharge Summaries | Variable |
| **medical_summarization** | Note Generation | Clinical Note Summarization | Variable |

### Gated Scenarios (Require authentication):

- EHRSHOT scenarios (require PhysioNet credentials)
- Other clinical datasets with restricted access

## 3. Running Multiple Scenarios

Let's set up a configuration to run multiple scenarios in sequence.

In [ ]:
import os

# Configuration
HOME_DIR = os.path.expanduser("~")
OUTPUT_PATH = os.path.join(HOME_DIR, "benchmark_output")
SUITE_NAME = "multi-scenario-workshop"
MAX_EVAL_INSTANCES = 5  # Small number for demo

# Define multiple scenarios to test
# Note: Adjust model deployments based on your setup
scenarios = [
    {
        "name": "PubMedQA",
        "entry": "pubmed_qa:model=openai/gpt-3.5-turbo",
        "description": "Medical literature QA"
    },
    # Add more scenarios as needed
    # {
    #     "name": "MedQA",
    #     "entry": "med_qa:model=openai/gpt-3.5-turbo",
    #     "description": "Medical exam questions"
    # },
]

print("Configured Scenarios:")
for i, scenario in enumerate(scenarios, 1):
    print(f"{i}. {scenario['name']}: {scenario['description']}")
    print(f"   Entry: {scenario['entry']}")
    print()

## 4. Understanding Scenario Output

Each scenario produces structured output including:

- **scenario_state.json**: Complete state of all requests and responses
- **scenario.json**: Metadata about the scenario
- **stats.json**: Performance statistics and metrics
- **instances.json**: Individual test instances

Let's examine the structure of a typical scenario output.

In [ ]:
import json
import glob

# Look for existing scenario runs
runs_base = os.path.join(OUTPUT_PATH, "runs")

if os.path.exists(runs_base):
    # Find all stats.json files
    stats_files = glob.glob(os.path.join(runs_base, "**", "stats.json"), recursive=True)
    
    if stats_files:
        print(f"Found {len(stats_files)} run(s) with statistics\n")
        
        # Examine first stats file
        stats_file = stats_files[0]
        print(f"Examining: {stats_file}")
        print("=" * 60)
        
        with open(stats_file, 'r') as f:
            stats = json.load(f)
        
        print("\nAvailable metrics:")
        for i, stat in enumerate(stats[:10], 1):  # Show first 10
            name = stat.get('name', {}).get('name', 'Unknown')
            value = stat.get('sum', stat.get('mean', 'N/A'))
            print(f"{i}. {name}: {value}")
        
        if len(stats) > 10:
            print(f"... and {len(stats) - 10} more metrics")
    else:
        print("No stats files found. Run an evaluation first.")
else:
    print(f"Runs directory not found: {runs_base}")
    print("Run an evaluation first to generate results.")

## 5. Metrics in MedHELM

MedHELM uses various metrics depending on the task:

### Accuracy Metrics:
- **Exact Match**: Strict string matching
- **F1 Score**: Token-level overlap
- **Accuracy**: Correct predictions

### Summarization Metrics:
- **ROUGE-1, ROUGE-2, ROUGE-L**: N-gram overlap
- **BERTScore**: Semantic similarity

### LLM-as-Judge Metrics:
- **Jury Score**: Multi-model evaluation with rubrics
- Custom criteria: Faithfulness, Safety, Clarity, etc.

## 6. Comparing Scenarios

Let's create a comparison table of results across different scenarios.

In [ ]:
import json
import glob
import pandas as pd

def extract_key_metrics(stats_file):
    """Extract key performance metrics from a stats file."""
    with open(stats_file, 'r') as f:
        stats = json.load(f)
    
    metrics = {}
    for stat in stats:
        name = stat.get('name', {}).get('name', '')
        # Extract common metrics
        if 'exact_match' in name.lower():
            metrics['Exact Match'] = stat.get('mean', stat.get('sum', 0))
        elif 'accuracy' in name.lower():
            metrics['Accuracy'] = stat.get('mean', stat.get('sum', 0))
        elif 'rouge' in name.lower() and 'rouge_l' in name.lower():
            metrics['ROUGE-L'] = stat.get('mean', stat.get('sum', 0))
    
    return metrics

# Find all stats files across suites
if os.path.exists(runs_base):
    stats_files = glob.glob(os.path.join(runs_base, "**", "stats.json"), recursive=True)
    
    if len(stats_files) > 0:
        comparison_data = []
        
        for stats_file in stats_files:
            # Extract scenario name from path
            parts = stats_file.split(os.sep)
            suite = parts[-3] if len(parts) > 3 else 'Unknown'
            scenario = parts[-2] if len(parts) > 2 else 'Unknown'
            
            metrics = extract_key_metrics(stats_file)
            
            if metrics:
                comparison_data.append({
                    'Suite': suite,
                    'Scenario': scenario,
                    **metrics
                })
        
        if comparison_data:
            df = pd.DataFrame(comparison_data)
            print("\nScenario Comparison:")
            print("=" * 60)
            print(df.to_string(index=False))
        else:
            print("No comparable metrics found in available runs.")
    else:
        print("No evaluation results found yet.")
else:
    print("No runs directory found.")

## 7. Access Levels for Scenarios

MedHELM scenarios have different access levels:

### Public Scenarios
- Available to everyone
- No authentication required
- Use `run_entries_medhelm_public.conf`

### Gated Scenarios
- Require credentials (e.g., PhysioNet)
- Need to apply for access
- Use `run_entries_medhelm_gated.conf`

### Private Scenarios
- Organization-specific datasets
- Restricted to authorized members
- Use `run_entries_medhelm_private_{org}.conf`

## Summary

In this notebook, you learned:
1. ✓ The MedHELM clinician-validated taxonomy
2. ✓ Available scenarios across different categories
3. ✓ How to configure and run multiple scenarios
4. ✓ Understanding scenario outputs and metrics
5. ✓ Comparing results across scenarios
6. ✓ Access levels for different scenarios

### Next Steps:

- Create custom benchmarks in `workshop-notebook-4-custom-benchmarks.ipynb`
- Run full evaluations with more instances
- Apply for gated dataset access if needed
- Explore the official MedHELM leaderboard